# Задача 1 — SFT: перенос простого стиля

Дообучаем `Qwen/Qwen3-4B-Instruct-2507` через QLoRA на датасете `kid_adult`. Цель — перенести простой стиль ответа и измерить средний `P_simple` на `public_test_style`.


### Зависимости

In [ ]:
%pip install -q -U "transformers>=4.53.0" "accelerate>=1.8.0" "peft>=0.16.0" "bitsandbytes>=0.46.0" scipy scikit-learn safetensors tqdm


### Импорты и сиды
Фиксируем всё для воспроизводимости. tf32 на амперах ускоряет матричные умножения без заметной потери точности.

In [ ]:
import json, os, random
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np, torch
from scipy.sparse import hstack
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    get_linear_schedule_with_warmup, set_seed
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
DATA_DIR = Path("task_data")
STYLE_CLF_PATH = DATA_DIR / "style_clf.pkl"
ADAPTER_DIR = Path("adapters/task1_sft")
os.environ["HF_HUB_DISABLE_XET"] = "1"

def fix_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    torch.backends.cuda.matmul.allow_tf32 = True

fix_seeds()
print("Seed:", SEED)
print("CUDA available:", torch.cuda.is_available())


### Данные
Скачиваем с GitHub, если локально нет.

In [ ]:
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/Dom82209/ml-olympiad-red/main/task_data"

def download_if_missing(filename):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    path = DATA_DIR / filename
    if path.exists():
        return path
    url = f"{GITHUB_RAW_BASE}/{filename}"
    print(f"Downloading {filename}...")
    urlretrieve(url, path)
    return path

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

download_if_missing("kid_adult.jsonl")
download_if_missing("public_test_style.jsonl")
download_if_missing("style_clf.pkl")

train_data = read_jsonl(DATA_DIR / "kid_adult.jsonl")
test_data = read_jsonl(DATA_DIR / "public_test_style.jsonl")

print(f"Loaded: train={len(train_data)}, test={len(test_data)}")
print("Example keys:", train_data[0].keys())


### Токенайзер
Настраиваем чат-шаблон. padding_side="right" — обязательно для causal LM, иначе attention mask ломается.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

def fmt_prompt(p):
    return tok.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False, add_generation_prompt=True
    )

def fmt_full(p, a):
    return tok.apply_chat_template(
        [{"role": "user", "content": p}, {"role": "assistant", "content": a}],
        tokenize=False, add_generation_prompt=False
    )

### Датасет
Маскируем промпт (-100), лосс только по ответу.

In [ ]:
class SFTDataset(Dataset):
    def __init__(self, rows, tok, max_len=512):
        self.items = []
        for row in rows:
            p_ids = tok(fmt_prompt(row["prompt"]), add_special_tokens=False, truncation=True, max_length=max_len)["input_ids"]
            full_ids = tok(fmt_full(row["prompt"], row["kid"]), add_special_tokens=False, truncation=True, max_length=max_len)["input_ids"]

            labels = full_ids.copy()
            p_len = min(len(p_ids), len(labels))
            labels[:p_len] = [-100] * p_len # игнорим промпт при лоссе

            self.items.append({"input_ids": full_ids, "labels": labels})

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    inp, mask, lbl = [], [], []
    for x in batch:
        pad = max_len - len(x["input_ids"])
        inp.append(x["input_ids"] + [tok.pad_token_id] * pad)
        mask.append([1] * len(x["input_ids"]) + [0] * pad)
        lbl.append(x["labels"] + [-100] * pad)

    return {
        "input_ids": torch.tensor(inp),
        "attention_mask": torch.tensor(mask),
        "labels": torch.tensor(lbl)
    }

dataset = SFTDataset(train_data, tok)
print("Dataset size:", len(dataset))

### Модель и LoRA
Загружаем в 4-битном кванте (nf4), навешиваем LoRA-адаптеры на все линейные проекции. use_cache=False — без этого градиенты считаются некорректно при обучении.

In [ ]:
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_cfg, device_map="auto", trust_remote_code=True
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

### Обучение
Ручной трейн-луп. grad_acc=8 чтобы эффективный батч был 8 при реальном bs=1 (иначе в 24GB VRAM не влезает). Сохраняем только адаптеры — base-модель не трогаем.

In [ ]:
fix_seeds()
epochs, bs, grad_acc, lr = 1, 1, 8, 2e-4

loader = DataLoader(dataset, batch_size=bs, shuffle=True, collate_fn=collate_fn)
opt = torch.optim.AdamW(model.parameters(), lr=lr)
total_steps = max(1, epochs * len(loader) // grad_acc)
sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=max(1, int(total_steps * 0.03)), num_training_steps=total_steps)

model.train()
opt.zero_grad(set_to_none=True)

for epoch in range(epochs):
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for step, batch in enumerate(pbar):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        loss = model(**batch).loss / grad_acc
        loss.backward()

        if (step + 1) % grad_acc == 0:
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)

        pbar.set_postfix(loss=float(loss.detach().cpu()) * grad_acc)

ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
print("Saved adapters to", ADAPTER_DIR)

### Генерация на тесте
do_sample=False — по условию задачи нужна детерминированная генерация. При декодировании обрезаем входной промпт, оставляем только ответ модели.

In [ ]:
@torch.no_grad()
def generate(p, max_new=160):
    model.eval()
    inputs = tok(fmt_prompt(p), return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new, do_sample=False,
        pad_token_id=tok.eos_token_id, eos_token_id=tok.eos_token_id
    )
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

answers = [generate(row["prompt"]) for row in tqdm(test_data, desc="Generating")]
print("Generated:", len(answers))
print("Example:", answers[0][:100])

### Метрика
Считаем P_simple через предобученный клф. Пороги для буквы — из условия.

In [ ]:
import pickle

with open(STYLE_CLF_PATH, "rb") as f:
    obj = pickle.load(f)

clf, (w_vec, c_vec) = obj["clf"], obj["vecs"]
simple_idx = list(clf.classes_).index(1)

def get_p_simple(texts):
    feats = hstack([w_vec.transform(texts), c_vec.transform(texts)])
    return clf.predict_proba(feats)[:, simple_idx]

scores = get_p_simple(answers)
metric = float(np.mean(scores))

print(f"Task 1 SFT P_simple = {metric:.6f}")
print(f"Min P_simple = {np.min(scores):.6f}, max = {np.max(scores):.6f}")

def get_letter(v):
    if v < 0.4: return "А"
    if v < 0.7: return "Б"
    if v < 0.9: return "В"
    return "Г"

print(f"Task 1 answer interval letter = {get_letter(metric)}")